# RouteAlert — เทรน Siren / Non-Siren Audio Detector

เทรนโมเดลจำแนก **เสียงไซเรนรถพยาบาล vs เสียงทั่วไป** จาก spectrogram แล้ว export เป็น TFLite
ให้ตรงกับ `lib/core/ml/siren_detection_service.dart` ในแอป Flutter **แบบเป๊ะๆ**

**สำคัญที่สุด — รูปทรง feature ต้องตรงกันทั้งสองฝั่ง:**
ไฟล์นี้คำนวณ log-magnitude spectrogram ด้วยอัลกอริทึมเดียวกับที่ `siren_detection_service.dart`
ใช้บนอุปกรณ์จริงทุกขั้นตอน (sample rate, FFT size, hop size, จำนวน time steps, จำนวน freq bins,
log-spaced binning, log compression, min-max normalization) ถ้าคุณแก้ค่าคงที่ฝั่งใดฝั่งหนึ่ง
ต้องแก้อีกฝั่งให้ตรงกันด้วย ไม่งั้นโมเดลที่เทรนได้จะทำนายผิดเพี้ยนตอนใช้งานจริงในแอป (input mismatch)

**ค่าคงที่ (อ้างอิงจาก `SirenDetectionService` ในไฟล์ Dart):**
```
SAMPLE_RATE   = 16000   # Hz
CLIP_SECONDS  = 1.0     # วินาทีต่อคลิป
CLIP_SAMPLES  = 16000   # = SAMPLE_RATE * CLIP_SECONDS
FFT_SIZE      = 1024    # ต่อเฟรม (ต้องเป็นเลขยกกำลัง 2)
HOP_SIZE      = 512     # ระยะเลื่อนหน้าต่างระหว่างเฟรม (overlap 50%)
TIME_STEPS    = 32      # จำนวนเฟรมเวลาต่อคลิป (คงที่เสมอ)
FREQ_BINS     = 64      # จำนวนแถบความถี่แบบ log-spaced
INPUT_SHAPE   = [1, 32, 64, 1]   # [batch, time_steps, freq_bins, channel]
```

**เกี่ยวกับข้อมูล:** ตัวอย่างนี้ใช้ [UrbanSound8K](https://urbansounddataset.weebly.com/urbansound8k.html)
เป็นจุดเริ่มต้น (มีคลาส `siren` อยู่แล้ว ส่วนคลาสอื่นๆ ทั้งหมดใช้เป็น `not_siren`) เป็นชุดข้อมูลสาธารณะ
ทั่วไป **ไม่ใช่เสียงไซเรนรถพยาบาลไทยโดยเฉพาะ** — ถ้าจะใช้งานจริงในไทย แนะนำเก็บเสียงไซเรนรถพยาบาล/
ตำรวจ/ดับเพลิงในไทยเพิ่มเติมเองแล้วผสมเข้าไปใน `siren/` เพื่อความแม่นยำที่ดีขึ้น ไฟล์นี้**ไม่ได้อ้าง**
ตัวเลข accuracy ใดๆ ล่วงหน้า เพราะขึ้นกับข้อมูลจริงที่คุณใช้เทรน

**ก่อนรัน:** เตรียมโฟลเดอร์เสียง 2 โฟลเดอร์ `siren/` กับ `not_siren/` (ไฟล์ .wav ความยาวเท่าไหร่ก็ได้
โค้ดจะตัด/pad ให้เป็น 1 วินาทีให้เอง) บีบอัดเป็น `dataset.zip` โครงสร้าง:
```
dataset/
  siren/        (เสียงไซเรนรถพยาบาล/ฉุกเฉิน)
  not_siren/    (เสียงทั่วไป: รถยนต์, ถนน, พูดคุย, เพลง, เงียบ ฯลฯ)
```

In [ ]:
# ==== 1. เตรียมสภาพแวดล้อม ====
!pip install -q tensorflow librosa soundfile

In [ ]:
# ==== 2. อัปโหลด dataset.zip ====
# โครงสร้างที่ต้องมีในไฟล์ zip:
#   dataset/
#     siren/        (เสียงไซเรนรถพยาบาล/ฉุกเฉิน .wav)
#     not_siren/    (เสียงทั่วไปที่ไม่ใช่ไซเรน .wav)
#
# ถ้าใช้ UrbanSound8K: แยกไฟล์ที่ classID == 8 ("siren") ไปไว้ใน siren/
# ที่เหลือทั้งหมด (10 คลาส ลบ siren) ไปไว้ใน not_siren/ ก่อนบีบอัดเป็น dataset.zip
from google.colab import files
uploaded = files.upload()  # เลือก dataset.zip ที่เตรียมไว้

import zipfile
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [ ]:
# ==== 3. ค่าคงที่ Spectrogram — ต้องตรงกับ siren_detection_service.dart เป๊ะๆ ====
import numpy as np

SAMPLE_RATE = 16000
CLIP_SECONDS = 1.0
CLIP_SAMPLES = 16000
FFT_SIZE = 1024
HOP_SIZE = 512
TIME_STEPS = 32
FREQ_BINS = 64


def hann_window(size):
    n = np.arange(size)
    return 0.5 - 0.5 * np.cos(2.0 * np.pi * n / (size - 1))


_WINDOW = hann_window(FFT_SIZE)


def bin_to_log_freq_bins(linear_mags):
    """พอร์ตตรงจาก _binToLogFreqBins() ใน siren_detection_service.dart"""
    nyquist_bins = FFT_SIZE // 2
    result = np.zeros(FREQ_BINS, dtype=np.float64)
    max_log = np.log(float(nyquist_bins))

    for b in range(FREQ_BINS):
        lo_edge = np.exp(max_log * b / FREQ_BINS)
        hi_edge = np.exp(max_log * (b + 1) / FREQ_BINS)
        lo = int(np.clip(np.floor(lo_edge), 1, nyquist_bins - 1))
        hi = int(np.clip(np.floor(hi_edge), lo + 1, nyquist_bins))
        segment = linear_mags[lo:hi]
        result[b] = segment.mean() if segment.size > 0 else 0.0

    # Log compress (dB-like)
    result = np.log(result + 1e-6)

    # Normalize 0..1 ต่อคลิป (min-max)
    min_val = result.min()
    max_val = result.max()
    rng = 1.0 if abs(max_val - min_val) < 1e-6 else (max_val - min_val)
    result = np.clip((result - min_val) / rng, 0.0, 1.0)
    return result


def compute_log_spectrogram(samples):
    """พอร์ตตรงจาก computeLogSpectrogram() ใน siren_detection_service.dart
    คืนค่า array shape (TIME_STEPS, FREQ_BINS) — ค่าเดียวกับที่แอป Flutter
    จะป้อนเข้าโมเดลตอนใช้งานจริงบนอุปกรณ์
    """
    padded = np.zeros(CLIP_SAMPLES, dtype=np.float64)
    copy_len = min(len(samples), CLIP_SAMPLES)
    padded[:copy_len] = samples[:copy_len]

    frames = []
    offset = 0
    while len(frames) < TIME_STEPS:
        frame = np.zeros(FFT_SIZE, dtype=np.float64)
        end = min(offset + FFT_SIZE, len(padded))
        if offset < len(padded):
            frame[: end - offset] = padded[offset:end]
        frame = frame * _WINDOW

        spectrum = np.fft.rfft(frame, n=FFT_SIZE)  # length FFT_SIZE/2 + 1
        mags_full = np.zeros(FFT_SIZE, dtype=np.float64)
        mags_full[: len(spectrum)] = np.abs(spectrum)
        # ให้ตรงกับ Dart: fftea.realFft คืน Float64x2List ยาว FFT_SIZE เต็ม (conjugate
        # symmetric) แล้วเราใช้ magnitudes()[0:FFT_SIZE] ทั้งก้อน แต่ bin_to_log_freq_bins
        # อ่านแค่ index 1..nyquist_bins-1 (ครึ่งแรกที่ไม่ซ้ำ) จึงเทียบเท่ากับ np.fft.rfft ตรงนี้
        frames.append(bin_to_log_freq_bins(mags_full))
        offset += HOP_SIZE

    return np.stack(frames, axis=0)  # (TIME_STEPS, FREQ_BINS)


print('พร้อมคำนวณ spectrogram รูปทรง:', (TIME_STEPS, FREQ_BINS))

In [ ]:
# ==== 4. โหลดไฟล์เสียงทั้งหมด แปลงเป็น spectrogram ล่วงหน้า (dataset เล็กพอที่จะเก็บใน RAM) ====
import glob
import os

import librosa


def load_folder(folder, label):
    X, y = [], []
    paths = glob.glob(os.path.join(folder, '*'))
    for p in paths:
        try:
            samples, _sr = librosa.load(p, sr=SAMPLE_RATE, mono=True)
        except Exception as e:
            print('ข้ามไฟล์ที่อ่านไม่ได้:', p, e)
            continue
        spec = compute_log_spectrogram(samples)
        X.append(spec)
        y.append(label)
    return X, y


X_siren, y_siren = load_folder('dataset/siren', 1)
X_not, y_not = load_folder('dataset/not_siren', 0)

print(f'siren: {len(X_siren)} ไฟล์, not_siren: {len(X_not)} ไฟล์')

X = np.array(X_siren + X_not, dtype=np.float32)[..., np.newaxis]  # (N, 32, 64, 1)
y = np.array(y_siren + y_not, dtype=np.float32)
print('X shape:', X.shape, '  y shape:', y.shape)

In [ ]:
# ==== 5. แบ่ง train/validation ====
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('train:', X_train.shape, ' val:', X_val.shape)

In [ ]:
# ==== 6. สร้างโมเดล — CNN ขนาดเล็ก (input เป็น spectrogram ไม่ใช่ภาพถ่าย 224x224
# จึงไม่ใช้ transfer learning จาก ImageNet เหมือน accident/anti-spoofing classifier) ====
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TIME_STEPS, FREQ_BINS, 1)),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    # Sigmoid เดี่ยว: output[0][0] = ความมั่นใจว่าเป็นไซเรน (0.0-1.0)
    # ให้ตรงกับที่ siren_detection_service.dart อ่านค่า output[0][0] ตรงๆ ไม่ผ่าน softmax/argmax
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# ==== 7. เทรน ====
EPOCHS = 25
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=16,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)

In [ ]:
# ==== 8. ตรวจผลลัพธ์คร่าวๆ ก่อน export (ไม่ fix ตัวเลขไว้ล่วงหน้า ขึ้นกับ dataset จริงที่ใช้) ====
loss, acc = model.evaluate(X_val, y_val)
print(f'Validation accuracy: {acc*100:.1f}%')

In [ ]:
# ==== 9. Export เป็น TFLite แบบ quantized (ให้ตรงกับที่ siren_detection_service.dart คาดไว้) ====
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization ลดขนาดไฟล์
tflite_model = converter.convert()

with open('siren_detector.tflite', 'wb') as f:
    f.write(tflite_model)

print('Export เสร็จแล้ว: siren_detector.tflite')
print('Input shape ที่โมเดลนี้คาดหวัง: [1, 32, 64, 1]  (ตรงกับ kInputShape ใน Dart)')
print('Output shape: [1, 1]  (sigmoid confidence เดี่ยว — ไม่ใช่ softmax หลายคลาส)')

In [ ]:
# ==== 10. ดาวน์โหลดไฟล์ ====
# นำไฟล์ไปวางที่:
#   assets/models/siren_detector.tflite
# ในโปรเจกต์ Flutter (โฟลเดอร์ assets/models/ ถูกประกาศไว้ใน pubspec.yaml แล้วทั้งโฟลเดอร์
# ด้วย wildcard "assets/models/" จึงไม่ต้องแก้ pubspec.yaml เพิ่ม) แล้วรันแอปได้เลย
# ไม่ต้องแก้โค้ด Dart อะไรเพิ่ม — SirenDetectionService จะโหลดไฟล์นี้อัตโนมัติตอน initialize()
from google.colab import files
files.download('siren_detector.tflite')